# VLM cross-modal unlearning — Stage 1 runner

Thin runner: every cell calls one script; no experiment logic here. All outputs go to
`/content/drive/MyDrive/vlm_unlearn/`. Every job is **resumable**: after a disconnect, re-run the
setup cells (1a–1c) and then the job cell you were on. Order and runtimes: `README.md`.
Protocol: `PROTOCOL.md` + `DEVIATIONS.md`.

## 1. Setup (run every session)

In [ ]:
# --- set once ---------------------------------------------------------------
REPO_URL = "https://github.com/<you>/VLMUnlearn.git"   # private repo: https://<token>@github.com/<you>/VLMUnlearn.git
BRANCH = "main"
MODEL = "qwen2vl"   # "qwen2vl" = pipeline check only | "llava" = primary | "qwen25vl" = replication phase only
# ---------------------------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")
import os
os.environ["VLMU_ROOT"] = "/content/drive/MyDrive/vlm_unlearn"   # checkpoints, logs, results
os.environ["VLMU_LOCAL_CACHE"] = "/content/vlmu_cache"            # fast local copy of processed data
os.environ["HF_HOME"] = "/content/hf_home"                        # base model weights (local disk)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs(os.environ["VLMU_ROOT"], exist_ok=True)

In [ ]:
import os
if not os.path.exists("/content/VLMUnlearn"):
    !git clone -b {BRANCH} {REPO_URL} /content/VLMUnlearn
else:
    !git -C /content/VLMUnlearn pull --ff-only
%cd /content/VLMUnlearn
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, transformers, peft; print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.is_available())"

## 2. Data (once; CPU is fine) — ~10 min
Downloads MLLMU-Bench + MMBench to Drive, builds the frozen splits and paired items, runs the data checks.

In [ ]:
!python scripts/prepare_data.py --config configs/data.yaml

Optional: regenerate the E0 data audit (`analysis/data_audit.md`) from the Drive copy of the raw data.

In [ ]:
!python scripts/data_audit.py

## 3. Smoke test (once per model) — ~10–15 min
Runs every job below with `--debug` (~20 samples each) into `vlm_unlearn/debug/`; earlier debug outputs are archived, not deleted. The final sanity step is *expected* to report a failed gate (the debug vanilla trains for 6 steps). **Send me the output of this cell before running anything below.**

In [ ]:
!python scripts/smoke_test.py --model {MODEL}

---
**Cells 4–9 are the real runs. Do not start them until the protocol owner approves.**

## 4. Pretrained baseline (no fine-tuning: measures priors)

In [ ]:
!python scripts/eval_pretrained.py --config configs/{MODEL}/vanilla.yaml

## 5. Vanilla fine-tune (both forms) + full eval + gate (≥0.80 in both modalities) + frozen doubly-correct set
If the gate fails, unlearning is blocked — stop and report.

In [ ]:
!python scripts/finetune_vanilla.py --config configs/{MODEL}/vanilla.yaml
!cat /content/drive/MyDrive/vlm_unlearn/runs/{MODEL}/vanilla/gate.json

## 6. Milestone J on the real model (hook checks with a trained adapter) — a few minutes

In [ ]:
!python scripts/check_hooks.py --config configs/{MODEL}/grid.yaml

## 7. Unlearning grid (GA_Diff, NPO × text/image unlearning × 3 lr × 2 seeds = 24 runs)
Each run stops at its earliest eligible calibration checkpoint. Skips finished runs; re-run after a disconnect. Failed runs are logged in `results/run_log.csv` and the grid continues.

In [ ]:
!python scripts/run_grid.py --config configs/{MODEL}/grid.yaml

## 8. Selection (calibration only) + full evals of the selected checkpoints (confirmation entities)

In [ ]:
!python scripts/select_and_eval.py --config configs/{MODEL}/grid.yaml

## 9. Tables, figure, verdict + sanity checks

In [ ]:
!python scripts/aggregate.py --config configs/{MODEL}/grid.yaml
!python scripts/sanity.py --config configs/{MODEL}/grid.yaml
!cat /content/drive/MyDrive/vlm_unlearn/results/{MODEL}/verdict.json

## 10. (optional) push results to the repo

In [ ]:
!git add results analysis && git -c user.email=colab@local -c user.name=colab commit -m "results: {MODEL}" && git push